In [ ]:
#Load Dataset of Telco Churn
import pandas as pd    
df=pd.read_csv("Telco-Customer-Churn.csv")
#Shape of Dataset
print(df.shape)
#First five Rows 
print(df.head())
#Null Values
# print(df.isnull().sum())
# print(df['Churn'].value_counts())

(7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Co

In [8]:
#Data Cleaning
# Drop customerID as it's not useful for prediction
df.drop('customerID', axis=1, inplace=True)

Now separate Features and Target
# Define features (X) and target (y)

In [10]:
X = df.drop('Churn', axis=1)
# Convert Churn Column to binary
y = df['Churn'].map({'Yes': 1, 'No': 0})  
print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts())

Features shape: (7043, 19)
Target distribution:
 Churn
0    5174
1    1869
Name: count, dtype: int64


# Identify Column Types

In [11]:
# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']


C:\Users\Rashid\AppData\Local\Temp\ipykernel_2596\3833268852.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


# Split data for Training and Testing

In [12]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 5634
Test set size: 1409


# Build Preprocessing Pipeline

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Pipeline for numerical features
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Handle any missing values
    ('scaler', StandardScaler())
])

# Pipeline for categorical features
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_cols),
    ('categorical', categorical_pipeline, categorical_cols)
])

# Test the preprocessor
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
print(f"Processed training shape: {X_train_processed.shape}")

Processed training shape: (5634, 5304)


# Now wrap the preprocessor with different classifiers:

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Logistic Regression Pipeline
logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Random Forest Pipeline
random_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

# Now Train Models

In [15]:
# Train Logistic Regression
logistic_pipeline.fit(X_train, y_train)
print("Logistic Regression trained successfully!")

# Train Random Forest
random_pipeline.fit(X_train, y_train)
print("Random Forest trained successfully!")

Logistic Regression trained successfully!
Random Forest trained successfully!


# Now Evaluate Both Models

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n{'='*50}")
    print(f"{model_name} Performance")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    
    return y_pred, y_pred_proba

# Evaluate both models
lr_pred, lr_proba = evaluate_model(logistic_pipeline, X_test, y_test, "Logistic Regression")
rf_pred, rf_proba = evaluate_model(random_pipeline, X_test, y_test, "Random Forest")

c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



Logistic Regression Performance
Accuracy:  0.7935
Precision: 0.6293
Recall:    0.5401
F1-Score:  0.5813
ROC-AUC:   0.8403

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.84      0.89      0.86      1035
       Churn       0.63      0.54      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409



c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



Random Forest Performance
Accuracy:  0.7956
Precision: 0.6569
Recall:    0.4813
F1-Score:  0.5556
ROC-AUC:   0.8279

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.83      0.91      0.87      1035
       Churn       0.66      0.48      0.56       374

    accuracy                           0.80      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.80      0.78      1409



# Hyperparameter Tuning using GridSearchCV

In [19]:
from sklearn.model_selection import GridSearchCV

# Define parameter grids
lr_param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear', 'saga']
}

rf_param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, 15, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__class_weight': [None, 'balanced']
}

# Perform GridSearch for Logistic Regression
print("Performing GridSearch for Logistic Regression...")
logistic_grid_search = GridSearchCV(
    logistic_pipeline,
    lr_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
logistic_grid_search.fit(X_train, y_train)

print(f"\nBest Logistic Regression parameters: {logistic_grid_search.best_params_}")
print(f"Best CV score: {logistic_grid_search.best_score_:.4f}")

# Perform GridSearch for Random Forest
print("\nPerforming GridSearch for Random Forest...")
random_grid_search = GridSearchCV(
    random_pipeline,
    rf_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
random_grid_search.fit(X_train, y_train)

print(f"\nBest Random Forest parameters: {random_grid_search.best_params_}")
print(f"Best CV score: {random_grid_search.best_score_:.4f}")

Performing GridSearch for Logistic Regression...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



Best Logistic Regression parameters: {'classifier__C': 1, 'classifier__penalty': 'l2', 'classifier__solver': 'saga'}
Best CV score: 0.8451

Performing GridSearch for Random Forest...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits

Best Random Forest parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}
Best CV score: 0.8409


# Evaulate the tuned models

In [21]:
# Evaluate tuned models
print("\n" + "="*60)
print("EVALUATING TUNED MODELS")
print("="*60)

best_lr = logistic_grid_search.best_estimator_
best_rf = random_grid_search.best_estimator_

evaluate_model(best_lr, X_test, y_test, "Tuned Logistic Regression")
evaluate_model(best_rf, X_test, y_test, "Tuned Random Forest")


EVALUATING TUNED MODELS


c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



Tuned Logistic Regression Performance
Accuracy:  0.7935
Precision: 0.6301
Recall:    0.5374
F1-Score:  0.5801
ROC-AUC:   0.8403

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.84      0.89      0.86      1035
       Churn       0.63      0.54      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409



c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



Tuned Random Forest Performance
Accuracy:  0.7715
Precision: 0.5560
Recall:    0.6898
F1-Score:  0.6158
ROC-AUC:   0.8366

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.88      0.80      0.84      1035
       Churn       0.56      0.69      0.62       374

    accuracy                           0.77      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.79      0.77      0.78      1409



(array([0, 1, 0, ..., 0, 0, 0], shape=(1409,)),
 array([0.01724447, 0.74709416, 0.12403646, ..., 0.1784271 , 0.02368973,
        0.06479254], shape=(1409,)))

# Now save the best model using joblib

In [22]:
import joblib
from datetime import datetime

# Choose the best model based on test performance
# Let's compare ROC-AUC scores
y_pred_lr = best_lr.predict_proba(X_test)[:, 1]
y_pred_rf = best_rf.predict_proba(X_test)[:, 1]

lr_auc = roc_auc_score(y_test, y_pred_lr)
rf_auc = roc_auc_score(y_test, y_pred_rf)

print(f"Tuned Logistic Regression ROC-AUC: {lr_auc:.4f}")
print(f"Tuned Random Forest ROC-AUC: {rf_auc:.4f}")

# Select the best model
if rf_auc > lr_auc:
    best_model = best_rf
    model_name = "RandomForest"
else:
    best_model = best_lr
    model_name = "LogisticRegression"

print(f"\nSelected best model: {model_name}")

# Save the pipeline
model_filename = f'churn_prediction_pipeline_{model_name}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.joblib'
joblib.dump(best_model, model_filename)
print(f"Model saved as: {model_filename}")

# Also save as a generic name for production
joblib.dump(best_model, 'churn_prediction_pipeline_latest.joblib')
print("Latest model saved as: churn_prediction_pipeline_latest.joblib")

c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Tuned Logistic Regression ROC-AUC: 0.8403
Tuned Random Forest ROC-AUC: 0.8366

Selected best model: LogisticRegression
Model saved as: churn_prediction_pipeline_LogisticRegression_20260427_010156.joblib
Latest model saved as: churn_prediction_pipeline_latest.joblib


# Load and Test Saved Model

In [23]:
# Load the saved model
loaded_model = joblib.load('churn_prediction_pipeline_latest.joblib')

# Test on a single customer
sample_customer = X_test.iloc[[0]]
sample_pred = loaded_model.predict(sample_customer)[0]
sample_proba = loaded_model.predict_proba(sample_customer)[0]

print(f"Prediction for sample customer: {'Churn' if sample_pred == 1 else 'No Churn'}")
print(f"Probability (No Churn, Churn): {sample_proba}")

Prediction for sample customer: No Churn
Probability (No Churn, Churn): [0.96716455 0.03283545]


c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


# Prediction function for Production

In [24]:
def predict_churn(customer_data, model_path='churn_prediction_pipeline_latest.joblib'):
    """
    Predict churn for new customers.
    
    Parameters:
    -----------
    customer_data : pd.DataFrame or dict
        Customer information with same columns as training data
    model_path : str
        Path to saved pipeline
    
    Returns:
    --------
    predictions : np.array
        Predicted churn (1 = churn, 0 = no churn)
    probabilities : np.array
        Predicted probabilities
    """
    # Load model
    model = joblib.load(model_path)
    
    # Convert dict to DataFrame if needed
    if isinstance(customer_data, dict):
        customer_data = pd.DataFrame([customer_data])
    
    # Predict
    predictions = model.predict(customer_data)
    probabilities = model.predict_proba(customer_data)
    
    return predictions, probabilities

# Example usage
example_customer = X_test.iloc[0].to_dict()
pred, proba = predict_churn(example_customer)
print(f"Example prediction: {pred[0]}")
print(f"Example probabilities: {proba[0]}")

Example prediction: 0
Example probabilities: [0.96716455 0.03283545]


c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Rashid\anaconda3\envs\env_bert\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
